In [1]:
import pandas as pd
import numpy as np


np.random.seed(42)
n_samples = 1000

data = {
    'vehicle_count': np.random.randint(5, 150, n_samples),
    'avg_speed': np.random.uniform(5, 70, n_samples),
    'signal_wait_time': np.random.randint(0, 180, n_samples),
    'road_type': np.random.choice(['Highway', 'Arterial', 'Local'], n_samples),
    'weather': np.random.choice(['Clear', 'Rain', 'Fog', 'Snow'], n_samples),
    'accident': np.random.choice([0, 1], n_samples, p=[0.94, 0.06]),
    'hour_of_day': np.random.randint(0, 24, n_samples)
}

df = pd.DataFrame(data)

def determine_congestion(row):
    score = (row['vehicle_count'] * 0.4) + (row['signal_wait_time'] * 0.2) + (row['accident'] * 40)
    score -= (row['avg_speed'] * 0.5)
    if (7 <= row['hour_of_day'] <= 9) or (16 <= row['hour_of_day'] <= 19): score += 15
    return 'Low' if score < 15 else 'Medium' if score < 45 else 'High'

df['congestion_level'] = df.apply(determine_congestion, axis=1)
df.to_csv('urban_traffic_data.csv', index=False)
print("File 'urban_traffic_data.csv' has been created successfully!")


File 'urban_traffic_data.csv' has been created successfully!


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score


df = pd.read_csv('urban_traffic_data.csv')


le_features = LabelEncoder()
df['road_type'] = le_features.fit_transform(df['road_type'])
df['weather'] = le_features.fit_transform(df['weather'])

X = df.drop('congestion_level', axis=1)
y = df['congestion_level']


le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)


rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(X_test))


xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train, y_train)
xgb_acc = accuracy_score(y_test, xgb.predict(X_test))

print(f"Random Forest Accuracy: {rf_acc:.2%}")
print(f"XGBoost Accuracy: {xgb_acc:.2%}")


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:30:14] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Random Forest Accuracy: 83.50%
XGBoost Accuracy: 87.00%


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder

try:
    file_path = '/content/urban_traffic_data.csv'
    df = pd.read_csv(file_path)
    print("Successfully loaded the dataset.")
except FileNotFoundError:
    print("File not found. Please upload 'urban_traffic_data.csv' to the /content/ folder.")


target_col = [c for c in df.columns if 'congestion' in c.lower()][0]


le = LabelEncoder()
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col].astype(str))

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print("\n--- RANDOM FOREST PERFORMANCE ---")
print(f"Accuracy: {accuracy_score(y_test, rf_preds):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, rf_preds))


xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

print("\n--- XGBOOST (BOOSTING) PERFORMANCE ---")
print(f"Accuracy: {accuracy_score(y_test, xgb_preds):.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, xgb_preds))


Successfully loaded the dataset.

--- RANDOM FOREST PERFORMANCE ---
Accuracy: 0.8350
Confusion Matrix:
[[58  0 11]
 [ 0 35 13]
 [ 3  6 74]]

--- XGBOOST (BOOSTING) PERFORMANCE ---
Accuracy: 0.8700
Confusion Matrix:
[[63  1  5]
 [ 0 40  8]
 [ 5  7 71]]


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:33:51] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
